In [1]:
import pandas as pd
import numpy as np

In [2]:
port_cluster_data = pd.read_excel( "Port_Cluster.xlsx")
 

In [3]:
port_cluster_data.head()

,mean_draft,ae_t_steaming,aux_running,blr_running,activity_time,Cluster
0,6.80,3.8,3,1,3.0,1
1,6.50,1.9,3,1,1.0,1
2,7.15,7.5,3,1,6.0,1
3,7.60,8.0,3,1,6.9,1
4,7.60,13.0,3,1,12.2,1


In [4]:
port_cluster_data.max()

mean_draft       11.55
ae_t_steaming    48.50
aux_running       4.00
blr_running       1.00
activity_time    24.00
Cluster           1.00
dtype: float64

In [9]:
port_cluster_data.shape 

(296, 6)

In [11]:
port_cluster_data['activity_time'].sum()/24

134.59583333333333

# Port Data Profiles

In [15]:
def generate_port_profiles(cluster_data, 
                      no_of_profiles,
                      user_draft_range=2,
                      user_ae_time_range=4,
                      user_activity_time_range=4
                     ):
    
    global finaldict
    finaldict = {}
    
    for index in range(1, no_of_profiles + 1):
      
        # Filter data for the cluster
        data = cluster_data[cluster_data['Cluster'] == index][['mean_draft', 'ae_t_steaming', 'aux_running', 
                                                               'blr_running', 'activity_time']].copy()
        
        # Convert columns to numeric and round them
        data['aux_running'] = pd.to_numeric(data['aux_running'], errors='coerce').round(0)
        data['mean_draft'] = pd.to_numeric(data['mean_draft'], errors='coerce').round(0)

        # Create a pivot table for aux_running and draft data
        melt = pd.melt(data, id_vars=['mean_draft'], value_vars=['aux_running'])
        melt['aux_running'] = melt['value']
        pd2 = (melt.pivot_table(index='aux_running', columns='mean_draft', values='value', aggfunc=len, fill_value=0)
               .reset_index()
               .rename_axis(None, axis=1))
        pd2 = pd2.set_index('aux_running')
        pd2.index.name = None
        total = np.sum(pd2.to_numpy())
        pd2 = round((pd2 * 100 / total), 2)
        pd3 = pd2.reset_index()

        # Rename the column
        pd3.rename(columns={'index': 'AE_Running/Draft'}, inplace=True)
        pd3['Draft'] = pd.Series(pd3.columns[1:])

        # Get draft and AE running percentages
        mean_draft_percent_df = (data['mean_draft'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Draft_%'] = pd3['Draft'].map(dict(zip(mean_draft_percent_df.index, mean_draft_percent_df)))

        aux_running_percent_df = (data['aux_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['AE_Running'] = pd3['AE_Running/Draft']
        pd3['AE_Running%'] = pd3['AE_Running/Draft'].map(dict(zip(aux_running_percent_df.index, aux_running_percent_df)))

        # Extend the DataFrame to match the desired length (72 rows)
        pd3 = pd3.reindex(range(72))

        # Create percentage columns for activity time and other parameters
        activity_time_df = (data['activity_time'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Activity_Time'] = pd.Series(sorted(data['activity_time'].round().value_counts().index))
        pd3['Activity_Time%'] = pd3['Activity_Time'].map(dict(zip(activity_time_df.index, activity_time_df)))

        ae_steam_time_df = (data['ae_t_steaming'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['AE_Time'] = pd.Series(sorted(data['ae_t_steaming'].round().value_counts().index))
        pd3['AE_Time%'] = pd3['AE_Time'].map(dict(zip(ae_steam_time_df.index, ae_steam_time_df)))

        blr_running_df = (data['blr_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['BLR_Running'] = pd.Series(sorted(data['blr_running'].round().value_counts().index))
        pd3['BLR_Running%'] = pd3['BLR_Running'].map(dict(zip(blr_running_df.index, blr_running_df)))

        # Create range DataFrame
        range_df = pd.DataFrame(index=range(10))

        # Process ranges for draft, AE time, and activity time
        data_profile = pd3.iloc[:, -10:].copy()

        data_profile['draft_range'] = pd.cut(data_profile['Draft'], bins=list(range(0, int(data_profile['Draft'].max()) + user_draft_range, user_draft_range))).astype(str)
        range_profile_draft = data_profile.groupby('draft_range')['Draft_%'].sum().reset_index()
        range_df['Draft_Range'] = range_profile_draft['draft_range']
        range_df['Draft_%'] = range_profile_draft['Draft_%']

        data_profile['ae_time_range'] = pd.cut(data_profile['AE_Time'], bins=list(range(0, int(data_profile['AE_Time'].max()) + user_ae_time_range, user_ae_time_range))).astype(str)
        range_profile_ae_time = data_profile.groupby('ae_time_range')['AE_Time%'].sum().reset_index()
        range_df['AE_Time_Range'] = range_profile_ae_time['ae_time_range']
        range_df['AE_Time%'] = range_profile_ae_time['AE_Time%']

        data_profile['activity_time_range'] = pd.cut(data_profile['Activity_Time'], bins=list(range(0, int(data_profile['Activity_Time'].max()) + user_activity_time_range, user_activity_time_range))).astype(str)
        range_profile_activity_time = data_profile.groupby('activity_time_range')['Activity_Time%'].sum().reset_index()
        range_df['Activity_Time_Range'] = range_profile_activity_time['activity_time_range']
        range_df['Activity_Time%'] = range_profile_activity_time['Activity_Time%']

        # Prepare final profiles
        ae_draft_profile = pd3.iloc[:, :-10].dropna()
        parameters_range_profile = range_df.round(2).fillna('')
        parameters_profile = data_profile.round(2).iloc[:, :-4].fillna('')

        # Add profiles to the final dictionary
        finaldict[index] = {'ae_draft_profile': ae_draft_profile,
                            'parameters_profile': parameters_profile,
                            'parameters_range_profile': parameters_range_profile}

    return finaldict


In [33]:
generate_port_profiles(port_cluster_data,
                  1,
                  user_draft_range = 2, 
                  user_ae_time_range = 4,
                 user_activity_time_range=4
                 )




{1: {'ae_draft_profile':    AE_Running/Draft    5.0   6.0   7.0    8.0    9.0  10.0  11.0  12.0
  0               0.0  18.65  0.00  5.96   0.00   0.26  0.00  0.26  0.00
  1               1.0   1.30  3.11  9.07  16.32  10.88  6.48  2.07  0.26
  2               2.0   0.52  1.81  2.33   2.33   1.55  1.81  0.78  0.00
  3               3.0   0.26  1.30  2.85   2.59   3.37  3.63  0.26  0.00,
  'parameters_profile':    Draft Draft_% AE_Running AE_Running% Activity_Time Activity_Time% AE_Time  \
  0    5.0    20.7        0.0        25.1           0.0            6.0     0.0   
  1    6.0     6.2        1.0        49.5           1.0            8.0     1.0   
  2    7.0    20.2        2.0        11.1           2.0            8.8     2.0   
  3    8.0    21.2        3.0        14.2           3.0            8.5     3.0   
  4                                                 4.0            9.8     4.0   
  ..   ...     ...        ...         ...           ...            ...     ...   
  67           

In [12]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np

# Initialize global variables
finaldict = {}
activity_time_df = pd.DataFrame()
ae_steam_time_df = pd.DataFrame()
ae_running_df = pd.DataFrame()
blr_running_df = pd.DataFrame()
range_df = pd.DataFrame()
data_profile = pd.DataFrame()


In [13]:
# Cell 2: Process Data for Cluster 1
index = 1
data = port_cluster_data[port_cluster_data['Cluster'] == index][[
    'mean_draft',
    'ae_t_steaming',
    'aux_running',
    'blr_running',
    'activity_time'
]].copy()

data['aux_running'] = data['aux_running'].apply(pd.to_numeric, args=('coerce',)).round(0)
data['mean_draft'] = data['mean_draft'].apply(pd.to_numeric, args=('coerce',)).round(0)

melt = pd.melt(data, id_vars=['mean_draft'], value_vars=['aux_running'])
melt['aux_running'] = melt['value']
pd2 = pd.DataFrame()
pd2 = melt.pivot_table(index='aux_running', columns='mean_draft', values='value', aggfunc=np.size, fill_value=0).reset_index().rename_axis(None, axis=1)
pd2 = pd2.set_index('aux_running')
pd2.index.name = None
total = np.sum(pd2.to_numpy())
pd2 = round((pd2 * 100 / total), 2)

pd3 = pd2.reset_index()
pd3.rename(columns={'index': 'AE_Running/Draft'}, inplace=True)
pd3['Draft'] = pd.Series(list(pd3.columns[1:]))  

mean_draft_percent_df = pd.DataFrame((data['mean_draft'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
pd3['Draft_%'] = pd3['Draft'].map(dict(zip(list(mean_draft_percent_df.index), mean_draft_percent_df['mean_draft'])))

aux_running_percent_df = pd.DataFrame((data['aux_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
pd3['AE_Running'] = pd3['AE_Running/Draft']
pd3['AE_Running%'] = pd3['AE_Running/Draft'].map(dict(zip(list(aux_running_percent_df.index), aux_running_percent_df['aux_running'])))

# Extend the DataFrame to match the series length
pd3 = pd3.reindex(range(72))

activity_time_df = pd.DataFrame((data['activity_time'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
pd3['Activity_Time'] = pd.Series(sorted(list(data['activity_time'].apply(lambda x: round(x)).value_counts().index)))
pd3['Activity_Time%'] = pd3['Activity_Time'].map(dict(zip(list(activity_time_df.index), activity_time_df['activity_time'])))

ae_steam_time_df = pd.DataFrame((data['ae_t_steaming'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
pd3['AE_Time'] = pd.Series(sorted(list(data['ae_t_steaming'].apply(lambda x: round(x)).value_counts().index)))
pd3['AE_Time%'] = pd3['AE_Time'].map(dict(zip(list(ae_steam_time_df.index), ae_steam_time_df['ae_t_steaming'])))

ae_running_df = pd.DataFrame((data['aux_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
pd3['AE_Running'] = pd.Series(sorted(list(data['aux_running'].apply(lambda x: round(x)).value_counts().index)))
pd3['AE_Running%'] = pd3['AE_Running'].map(dict(zip(list(ae_running_df.index), ae_running_df['aux_running'])))

blr_running_df = pd.DataFrame((data['blr_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
pd3['BLR_Running'] = pd.Series(sorted(list(data['blr_running'].apply(lambda x: round(x)).value_counts().index)))
pd3['BLR_Running%'] = pd3['BLR_Running'].map(dict(zip(list(blr_running_df.index), blr_running_df['blr_running'])))


KeyError: 'mean_draft'

In [24]:
# Cell 3: Create Range Profiles
range_df = pd.DataFrame(index=range(10))

data_profile = pd3.iloc[:, -14:]

data_profile['draft_range'] = pd.cut(x=data_profile['Draft'], bins=list(range(0, (int(data_profile['Draft'].max()) + int(data_profile['Draft'].max()) % 2 + 2), 2))).astype(str)
range_profile_draft = data_profile.bfill().groupby('draft_range')['Draft_%'].sum().reset_index()
range_df['Draft_Range'] = pd.Series(range_profile_draft['draft_range']).astype(str)
range_df['Draft_%'] = pd.Series(range_profile_draft['Draft_%'])

data_profile['activity_time_range'] = pd.cut(x=data_profile['Activity_Time'], bins=list(range(0, (int(data_profile['Activity_Time'].max()) + int(data_profile['Activity_Time'].max()) % 4 + 4), 4))).astype(str)
range_profile_activity_time = data_profile.bfill().groupby('activity_time_range')['Activity_Time%'].sum().reset_index()
range_df['Activity_Time_Range'] = pd.Series(range_profile_activity_time['activity_time_range']).astype(str)
range_df['Activity_Time%'] = pd.Series(range_profile_activity_time['Activity_Time%'])

data_profile['ae_time_range'] = pd.cut(x=data_profile['AE_Time'], bins=list(range(0, (int(data_profile['AE_Time'].max()) + int(data_profile['AE_Time'].max()) % 4 + 4), 4))).astype(str)
range_profile_ae_time = data_profile.bfill().groupby('ae_time_range')['AE_Time%'].sum().reset_index()
range_df['AE_Time_Range'] = pd.Series(range_profile_ae_time['ae_time_range']).astype(str)
range_df['AE_Time%'] = pd.Series(range_profile_ae_time['AE_Time%'])

user_ae_running_range = 1
data_profile['ae_running_range'] = pd.cut(x=data_profile['AE_Running'], bins=list(range(0, (int(data_profile['AE_Running'].max()) + int(data_profile['AE_Running'].max()) % 4 + 4), 4))).astype(str)

range_profile_ae_running = data_profile.bfill().groupby('ae_running_range')['AE_Running%'].sum().reset_index()
range_df['AE_Running_Range'] = pd.Series(range_profile_ae_running['ae_running_range']).astype(str)
range_df['AE_Running%'] = pd.Series(range_profile_ae_running['AE_Running%'])

# BLR Running range profile (if needed)
# range_df['BLR_Running'] = pd.Series(data_profile['BLR_Running'].values).astype(str)
# range_df['BLR_Running%'] = pd.Series(data_profile['BLR_Running%'])


In [25]:
# Cell 4: Store Results in Final Dictionary
port_aux_running_draft_profile = pd3.iloc[:, :-10].dropna()
port_parameters_range_profile = range_df.round(2).fillna('').replace('nan', "")
port_parameters_profile = data_profile.round(2).iloc[:, 4:-4].fillna('')

finaldict['port_aux_running_draft_profile_' + str(index)] = port_aux_running_draft_profile
finaldict['port_parameters_range_profile_' + str(index)] = port_parameters_range_profile
finaldict['port_parameters_profile_' + str(index)] = port_parameters_profile


In [26]:
port_aux_running_draft_profile

,AE_Running/Draft,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0
0,0.0,4.89,0.00,4.08,0.00,0.07,0.00,0.07,0.00
1,1.0,2.58,1.97,5.77,8.42,7.07,6.52,4.48,0.82
2,2.0,0.48,0.82,1.63,1.97,1.77,3.19,1.43,0.20
3,3.0,0.20,0.95,3.60,8.08,9.38,10.94,5.77,1.36
4,4.0,0.07,0.00,0.14,0.20,0.41,0.48,0.14,0.07


In [27]:
port_parameters_range_profile

,Draft_Range,Draft_%,Activity_Time_Range,Activity_Time%,AE_Time_Range,AE_Time%,AE_Running_Range,AE_Running%
0,"(4, 6]",11.9,"(0, 4]",15.9,"(0, 4]",12.5,"(0, 4]",90.9
1,"(6, 8]",33.9,"(12, 16]",12.8,"(12, 16]",13.7,,9.1
2,"(8, 10]",18.7,"(16, 20]",9.4,"(16, 20]",9.8,,
3,,0.0,"(20, 24]",27.1,"(20, 24]",15.6,,
4,,,"(24, 28]",0.1,"(24, 28]",3.9,,
5,,,"(28, 32]",0.1,"(28, 32]",1.2,,
6,,,"(4, 8]",17.7,"(32, 36]",0.6,,
7,,,"(8, 12]",14.9,"(36, 40]",0.5,,
8,,,,2.1,"(4, 8]",14.8,,
9,,,,,"(40, 44]",0.5,,


In [28]:
port_parameters_profile

,Draft,Draft_%,AE_Running,AE_Running%,Activity_Time,Activity_Time%,AE_Time,AE_Time%,BLR_Running,BLR_Running%
0,5.0,8.2,0.0,9.1,0.0,2.1,0.0,10.0,0.0,36.3
1,6.0,3.7,1.0,37.6,1.0,3.0,1.0,2.2,1.0,63.7
2,7.0,15.2,2.0,11.5,2.0,3.9,2.0,3.0,,
3,8.0,18.7,3.0,40.3,3.0,3.6,3.0,2.8,,
4,9.0,18.7,4.0,1.5,4.0,5.4,4.0,4.5,,
...,...,...,...,...,...,...,...,...,...,...
67,,,,,,,,,,
68,,,,,,,,,,
69,,,,,,,,,,
70,,,,,,,,,,


In [10]:
finaldict.keys()

dict_keys(['port_aux_running_draft_profile_1', 'port_parameters_range_profile_1', 'port_parameters_profile_1'])

# Find Subset Data Profile

In [35]:
subset_data_port = pd.read_excel("Subset_PORT_DF.xlsx")
subset_data_port

,mean_draft,ae_t_steaming,aux_running,blr_running,activity_time,Cluster
0,5.45,0.0,0,0,24.0,1
1,5.35,7.9,2,0,5.4,1
2,6.60,1.9,1,1,1.9,1
3,7.45,11.9,2,0,8.9,1
4,5.45,0.0,0,0,24.0,1
...,...,...,...,...,...,...
233,7.75,13.9,1,0,13.9,3
234,5.40,24.0,1,0,24.0,3
235,7.45,0.0,0,1,24.0,3
236,9.60,12.2,2,0,6.1,3


In [37]:
result= generate_port_profiles(subset_data_port, 
                      1,
                      user_draft_range = 2, 
 
                      user_ae_time_range = 4 ,
                      user_activity_time_range = 4, 
                     ) 

In [39]:
result

{1: {'ae_draft_profile':    AE_Running/Draft    5.0   6.0    7.0    8.0    9.0  10.0  11.0
  0               0.0  14.14  0.00   6.06   0.00   1.01  0.00  1.01
  1               1.0   2.02  4.04  11.11  15.15  10.10  5.05  5.05
  2               2.0   1.01  2.02   1.01   3.03   1.01  1.01  1.01
  3               3.0   1.01  2.02   6.06   1.01   1.01  4.04  0.00,
  'parameters_profile':    Draft Draft_% AE_Running AE_Running% Activity_Time Activity_Time% AE_Time  \
  0    5.0    18.2        0.0        22.2           0.0            6.1     0.0   
  1    6.0     8.1        1.0        52.5           1.0           11.1     1.0   
  2    7.0    24.2        2.0        10.1           2.0           10.1     2.0   
  3    8.0    19.2        3.0        15.2           3.0            8.1     3.0   
  4                                                 4.0           10.1     4.0   
  ..   ...     ...        ...         ...           ...            ...     ...   
  67                                    

In [41]:
finaldict.keys()

dict_keys([1])

In [21]:
for i in finaldict.keys():
 
    finaldict[i].to_excel( "Subset_Port_" + i +'.xlsx', index = False)

AttributeError: 'dict' object has no attribute 'to_excel'